# dummygrad on Colab

Builds dummygrad with CUDA, runs the test suite, then trains a small transformer.

**Runtime → Change runtime type → T4 GPU** (any NVIDIA GPU works), then **Runtime → Run all**.


In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
# Branch or tag to test
BRANCH = "main"

%cd /content
!rm -rf dummygrad && git clone -q --depth 1 -b {BRANCH} https://github.com/Keplercodes01/dummygrad.git
%cd /content/dummygrad
!git log --oneline -1

In [ ]:
# Configure + build (CUDA architecture is detected from the attached GPU)
!apt-get -qq install -y libopenblas-dev > /dev/null
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release 2>&1 | grep -E "CUDA|BLAS|Error"
!cmake --build build -j$(nproc) 2>&1 | grep -E "error|Error|Built target"

## Test suite

CPU gradient checks against finite differences, CUDA vs CPU parity, autograd engine tests, and end-to-end training. Ends with `N passed, 0 failed`.

In [ ]:
!./build/test_engine

## Training demo

Trains a 2-layer GQA transformer on a synthetic next-token task, saves a SafeTensors checkpoint, and reloads it. The loss should fall well below its starting value (~6.2).

In [ ]:
!cd build && ./train_colab